# Projeto #3: Performance Analysis

## Projeto #3: Performance Analysis

**Domínio:** Esportes / Performance de jogadores
**Pergunta:** Como está o jogador? Vai melhorar? Qual o valor de mercado?
**Conceitos cobertos:** Fundamentos + Prompting (W1-2), RAG histórico (W3),
Tool Use — stats e market data (W5), Agentic Loop streaming (W5-6), LangGraph
pipeline de tempo real (W6-7), Streaming/Real-time (W7), Confidence Scoring
de previsão de mercado (W7), Observability de latência de updates (W8)

In [ ]:
!pip install -q langgraph pydantic

import random
import time
from typing import Optional
from pydantic import BaseModel, Field

### 1. Schema + dados sintéticos (substituem os 500 players fictícios)

In [ ]:
class PlayerSnapshot(BaseModel):
    player_id: str
    age: int
    games_played: int
    goals: int
    assists: int
    minutes_per_game: float
    market_value_m: float  # em milhões

class PlayerAnalysis(BaseModel):
    player_id: str
    trend: str
    market_value_prediction_m: float
    confidence: float = Field(ge=0.0, le=1.0)
    rationale: str

def generate_players(n: int = 12, seed: int = 3) -> list[PlayerSnapshot]:
    random.seed(seed)
    out = []
    for i in range(n):
        out.append(PlayerSnapshot(
            player_id=f"player_{i:03d}",
            age=random.randint(18, 35),
            games_played=random.randint(5, 38),
            goals=random.randint(0, 25),
            assists=random.randint(0, 15),
            minutes_per_game=round(random.uniform(20, 90), 1),
            market_value_m=round(random.uniform(1, 80), 1),
        ))
    return out

players = generate_players()
print(f"✓ {len(players)} jogadores sintéticos")

### 2. RAG histórico (Week 3) — desempenho passado do mesmo jogador

In [ ]:
def retrieve_historical_seasons(player: PlayerSnapshot) -> list[dict]:
    """Simula busca de temporadas anteriores (substitua por query real
    ao seu data warehouse de estatísticas)."""
    random.seed(hash(player.player_id) % 1000)
    seasons = []
    for s in range(2):
        seasons.append({
            "season": f"202{3+s}",
            "goals": max(0, player.goals + random.randint(-5, 3)),
            "market_value_m": round(player.market_value_m * random.uniform(0.7, 1.1), 1),
        })
    return seasons

### 3. Tools — stats ao vivo e dados de mercado (Week 5)

In [ ]:
def tool_live_stats(player_id: str) -> dict:
    random.seed(hash(player_id) % 500)
    return {"player_id": player_id, "live_rating": round(random.uniform(5.5, 9.5), 1)}

def tool_market_comparables(player: PlayerSnapshot) -> dict:
    peer_avg = player.market_value_m * random.uniform(0.85, 1.15)
    return {"peer_avg_value_m": round(peer_avg, 1)}

### 4. "LLM" mockado com confidence scoring (Week 7)

In [ ]:
def mock_player_analysis(player: PlayerSnapshot, history: list[dict], live: dict, market: dict) -> PlayerAnalysis:
    """MOCK — troque por chamada real à Anthropic."""
    goal_trend = player.goals - (history[0]["goals"] if history else player.goals)
    trend = "subindo" if goal_trend > 0 else ("estável" if goal_trend == 0 else "caindo")

    predicted_value = (player.market_value_m + market["peer_avg_value_m"]) / 2
    if trend == "subindo":
        predicted_value *= 1.1
    elif trend == "caindo":
        predicted_value *= 0.9

    # confiança cai se temos poucos dados (poucos jogos) ou dado de mercado muito divergente
    data_quality = min(player.games_played / 20, 1.0)
    market_agreement = 1 - min(abs(predicted_value - player.market_value_m) / max(player.market_value_m, 1), 1)
    confidence = round(0.5 * data_quality + 0.5 * market_agreement, 2)

    return PlayerAnalysis(
        player_id=player.player_id,
        trend=trend,
        market_value_prediction_m=round(predicted_value, 1),
        confidence=confidence,
        rationale=f"live_rating={live['live_rating']}, peer_avg=${market['peer_avg_value_m']}M, trend={trend}",
    )

### 5. Grafo de streaming (Week 6-7): Ingest → Update → Report, em loop

Simula um pipeline de tempo real: a cada "tick" novos dados chegam (ex.: via
Cloud Pub/Sub em produção) e o grafo reprocessa o estado do jogador.

In [ ]:
from langgraph.graph import StateGraph, START, END

class LiveState(BaseModel):
    player: PlayerSnapshot
    history: list[dict] = []
    live: Optional[dict] = None
    market: Optional[dict] = None
    analysis: Optional[PlayerAnalysis] = None
    tick_latency_ms: float = 0

def node_ingest(state: LiveState) -> LiveState:
    t0 = time.time()
    state.history = retrieve_historical_seasons(state.player)
    state.live = tool_live_stats(state.player.player_id)
    state.market = tool_market_comparables(state.player)
    state.tick_latency_ms = (time.time() - t0) * 1000
    return state

def node_update(state: LiveState) -> LiveState:
    state.analysis = mock_player_analysis(state.player, state.history, state.live, state.market)
    return state

def node_report(state: LiveState) -> LiveState:
    return state

graph = StateGraph(LiveState)
graph.add_node("ingest", node_ingest)
graph.add_node("update", node_update)
graph.add_node("report", node_report)
graph.add_edge(START, "ingest")
graph.add_edge("ingest", "update")
graph.add_edge("update", "report")
graph.add_edge("report", END)

performance_agent = graph.compile()

### 6. Simulando 3 ticks de "tempo real" pra um jogador

In [ ]:
player = players[0]
print(f"📡 Streaming updates pra {player.player_id}\n")
for tick in range(3):
    result = performance_agent.invoke(LiveState(player=player))
    a = result["analysis"] if isinstance(result, dict) else result.analysis
    lat = result["tick_latency_ms"] if isinstance(result, dict) else result.tick_latency_ms
    print(f"[tick {tick+1}] trend={a.trend} valor_previsto=${a.market_value_prediction_m}M "
          f"confidence={a.confidence} (ingest_latency={lat:.1f}ms)")
    if a.confidence < 0.5:
        print("   ⚠️  confiança baixa — escalar pra analista humano")
    time.sleep(0.2)  # simula intervalo entre updates

**Próximos passos pra produção:**
- Trocar o loop de "ticks" por consumo real de Cloud Pub/Sub
- `tool_live_stats` → API real de dados ao vivo (Opta, StatsBomb, etc)
- Avaliação: comparar `market_value_prediction_m` vs valor real de mercado (Transfermarkt)
- Deploy: Cloud Run com endpoint de streaming (Server-Sent Events ou WebSocket)